# 🌧️ Data Storytelling: ปริมาณฝนและพื้นที่เสี่ยงน้ำท่วมของประเทศไทย

---
แหล่งที่มาข้อมูล: https://data.hii.or.th/

ชุดข้อมูลใน THackle: https://www.thackle.or.th/th/dataset/83


Google Colab: https://colab.research.google.com/drive/REMOVED




## 📋 ภาพรวม Notebook

| ขั้นตอน | รายละเอียด |
|---------|------------|
| **ไฟล์ 1** | `spatial-rain-hii.csv` — ข้อมูลปริมาณฝน (Min/Max/Avg) รายจังหวัด รายเดือน ปี 2018–2026 |
| **ไฟล์ 2** | `monthly-flood-risk-area.csv` — ข้อมูลพื้นที่เสี่ยงน้ำท่วมระดับตำบล จำแนกตามระดับความเสี่ยง 3 ระดับ |
| **ไฟล์ 3** | `province.csv` — **ไฟล์อ้างอิงชื่อจังหวัด** (ไทย / อังกฤษ / ภาค) |
| **เป้าหมาย** | วิเคราะห์แนวโน้มฝน ความเสี่ยงน้ำท่วม และความสัมพันธ์เชิงพื้นที่/เวลา |

---

## 🗺️ โครงสร้าง Notebook
1. Setup & Load Data
2. Province Name Normalization
3. Data Quality Audit
4. Data Cleaning
5. EDA เบื้องต้น
6. Advanced EDA & Insights
7. Dashboard Preparation


---
# 1️⃣ Setup & Load Data

### ติดตั้ง Library และโหลดข้อมูล

ติดตั้ง library ที่จำเป็น และโหลดทั้งสองไฟล์เข้ามาพร้อมตรวจสอบ schema เปรียบเทียบกัน


In [ ]:
# ติดตั้ง library ที่จำเป็น
!pip install plotly kaleido --quiet
!pip install pandas numpy matplotlib seaborn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')

# ─── ดาวน์โหลดและตั้งค่า Font TH Sarabun New ───
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
mpl.font_manager.fontManager.addfont('thsarabunnew-webfont.ttf')
mpl.rc('font', family='TH Sarabun New', size=16)
sns.set_theme(style='whitegrid', font='TH Sarabun New')
print('✅ โหลด Font TH Sarabun New สำเร็จ')

# ─── สี Theme ───
COLOR_BLUE   = '#1E88E5'
COLOR_ORANGE = '#FB8C00'
COLOR_GREEN  = '#43A047'
COLOR_RED    = '#E53935'
COLOR_PURPLE = '#8E24AA'

RISK_COLORS = {
    'เสี่ยงต่ำ':      '#43A047',
    'เสี่ยงปานกลาง':  '#FB8C00',
    'เสี่ยงสูง':      '#E53935'
}

print('\n✅ Import library สำเร็จทั้งหมด')


✅ โหลด Font TH Sarabun New สำเร็จ

✅ Import library สำเร็จทั้งหมด


In [ ]:
# ─── โหลดข้อมูล ───
# ** หากรันบน Google Colab ให้ Upload ไฟล์ทั้ง 3 ไฟล์ก่อน หรือแก้ path ด้านล่าง **

RAIN_PATH     = 'spatial-rain-hii.csv'
FLOOD_PATH    = 'monthly-flood-risk-area.csv'
PROVINCE_REF_PATH = 'province.csv'   # ← ไฟล์อ้างอิงชื่อจังหวัด

df_rain  = pd.read_csv(RAIN_PATH,  encoding='utf-8-sig')
df_flood = pd.read_csv(FLOOD_PATH, encoding='utf-8-sig')

print(f'📂 spatial-rain-hii.csv       : {df_rain.shape[0]:,} แถว × {df_rain.shape[1]} คอลัมน์')
print(f'📂 monthly-flood-risk-area.csv: {df_flood.shape[0]:,} แถว × {df_flood.shape[1]} คอลัมน์')
print(f'📂 province.csv               : จะโหลดในขั้นตอน Province Normalization')

📂 spatial-rain-hii.csv       : 7,546 แถว × 7 คอลัมน์
📂 monthly-flood-risk-area.csv: 27,023 แถว × 13 คอลัมน์
📂 province.csv               : จะโหลดในขั้นตอน Province Normalization


---
## 🗺️ Province Name Normalization

### ปรับชื่อจังหวัดให้ตรงกับไฟล์อ้างอิง `province.csv`

ตรวจพบปัญหาชื่อจังหวัดที่ไม่ตรงกันระหว่างไฟล์:

| ไฟล์ | ปัญหา | ตัวอย่าง |
|------|--------|----------|
| `monthly-flood-risk-area.csv` | ชื่อจังหวัดมีคำนำหน้า `จ.` | `จ.กรุงเทพมหานคร` → `กรุงเทพมหานคร` |

ขั้นตอนนี้จะ:
1. โหลด `province.csv` เป็น Reference (77 จังหวัด ครบถ้วน)
2. ตัด prefix `จ.` ออกจาก FLOOD dataset
3. ตรวจสอบและ Map ชื่อที่ยังไม่ตรง (ถ้ามี)
4. Merge ข้อมูล `Province (EN)` และ `ภาค` จาก Reference เข้าทุก dataset
5. แสดง Validation Report ยืนยันความครบถ้วน


In [ ]:
# ─── โหลด Province Reference ───
PROVINCE_REF_PATH = 'province.csv'
df_prov_ref = pd.read_csv(PROVINCE_REF_PATH, encoding='utf-8-sig')
print(f'📂 province.csv: {len(df_prov_ref)} จังหวัด')
print(df_prov_ref.head())

ref_names = set(df_prov_ref['จังหวัด'])
print(f'\n✅ ชื่อจังหวัดใน Reference: {len(ref_names)} จังหวัด')

📂 province.csv: 77 จังหวัด
         จังหวัด        Province    ภาค
0  กรุงเทพมหานคร         Bangkok   กลาง
1         กระบี่           Krabi    ใต้
2      กาญจนบุรี    Kanchanaburi   กลาง
3      กาฬสินธุ์         Kalasin  อีสาน
4      กำแพงเพชร  Kamphaeng Phet  เหนือ

✅ ชื่อจังหวัดใน Reference: 77 จังหวัด


In [ ]:
# ─── ตรวจสอบชื่อจังหวัดก่อนแก้ไข ───
print('=' * 60)
print('  🔍 Province Name Audit (ก่อนแก้ไข)')
print('=' * 60)

# --- RAIN ---
rain_names = set(df_rain['PROV_T'].unique())
rain_missing = sorted(rain_names - ref_names)
rain_ok      = sorted(rain_names & ref_names)
print(f'\n📊 RAIN — ชื่อจังหวัด:')
print(f'  ✅ ตรงกับ Reference : {len(rain_ok)} จังหวัด')
print(f'  ❌ ไม่ตรง (ต้องแก้)  : {len(rain_missing)} จังหวัด → {rain_missing}')

# --- FLOOD (ก่อน strip) ---
flood_names_raw = set(df_flood['PROV_T'].unique())
flood_missing_raw = sorted(flood_names_raw - ref_names)
print(f'\n📊 FLOOD — ชื่อจังหวัด (ก่อน strip จ.):')
print(f'  ✅ ตรงกับ Reference : {len(flood_names_raw & ref_names)} จังหวัด')
print(f'  ❌ ไม่ตรง          : {len(flood_missing_raw)} จังหวัด')
print(f'  ตัวอย่างที่ไม่ตรง  : {flood_missing_raw[:10]}')

  🔍 Province Name Audit (ก่อนแก้ไข)

📊 RAIN — ชื่อจังหวัด:
  ✅ ตรงกับ Reference : 77 จังหวัด
  ❌ ไม่ตรง (ต้องแก้)  : 0 จังหวัด → []

📊 FLOOD — ชื่อจังหวัด (ก่อน strip จ.):
  ✅ ตรงกับ Reference : 1 จังหวัด
  ❌ ไม่ตรง          : 74 จังหวัด
  ตัวอย่างที่ไม่ตรง  : ['จ.กระบี่', 'จ.กาญจนบุรี', 'จ.กาฬสินธุ์', 'จ.กำแพงเพชร', 'จ.ขอนแก่น', 'จ.จันทบุรี', 'จ.ฉะเชิงเทรา', 'จ.ชลบุรี', 'จ.ชัยนาท', 'จ.ชัยภูมิ']


In [ ]:
# ─── Manual Mapping: ชื่อที่ต้องแก้ด้วยมือ ───
# เพิ่ม mapping ที่นี่ถ้าพบชื่อที่ไม่ตรงในอนาคต
MANUAL_MAP = {
    # ชื่อในไฟล์ข้อมูล : ชื่อใน province.csv (Reference)
    # ตัวอย่าง: 'ชื่อเก่า': 'ชื่อใหม่',
}

# ชื่อที่ทราบว่าไม่มีใน Reference (ถ้ามี ให้ระบุไว้ที่นี่)
MISSING_FROM_REF = set()  # ปัจจุบันทุกจังหวัดมีใน province.csv ครบ 77 จังหวัดแล้ว

if MANUAL_MAP:
    print('📋 Manual Mapping ที่กำหนด:')
    for k, v in MANUAL_MAP.items():
        print(f'  {k!r} → {v!r}')
else:
    print('✅ ไม่มี Manual Mapping — ทุกจังหวัดในไฟล์ข้อมูลตรงกับ province.csv แล้ว')


✅ ไม่มี Manual Mapping — ทุกจังหวัดในไฟล์ข้อมูลตรงกับ province.csv แล้ว


In [ ]:
# ─── Apply Province Name Normalization ───
prov_fix_log = []

def normalize_prov_name(name, source_label):
    """ทำความสะอาดชื่อจังหวัด: strip จ., apply manual map"""
    # 1. ตัด prefix 'จ.' ออก
    stripped = str(name).strip()
    if stripped.startswith('จ.'):
        stripped = stripped[2:].strip()
    # 2. Apply manual map
    result = MANUAL_MAP.get(stripped, stripped)
    if result != name:
        prov_fix_log.append({'Dataset': source_label, 'ชื่อเดิม': name, 'ชื่อใหม่': result})
    return result

# ── แก้ไข RAIN ──
df_rain['PROV_T_orig'] = df_rain['PROV_T']  # เก็บต้นฉบับ
df_rain['PROV_T'] = df_rain['PROV_T'].apply(lambda x: normalize_prov_name(x, 'RAIN'))

# ── แก้ไข FLOOD ──
df_flood['PROV_T_orig'] = df_flood['PROV_T']  # เก็บต้นฉบับ
df_flood['PROV_T'] = df_flood['PROV_T'].apply(lambda x: normalize_prov_name(x, 'FLOOD'))

# ── สรุปการแก้ไข ──
df_fix_log = pd.DataFrame(prov_fix_log).drop_duplicates()
print(f'🧹 แก้ไขชื่อจังหวัดทั้งหมด: {len(df_fix_log)} รูปแบบ')
print()
print(df_fix_log.to_string(index=False))

🧹 แก้ไขชื่อจังหวัดทั้งหมด: 74 รูปแบบ

Dataset          ชื่อเดิม        ชื่อใหม่
  FLOOD         จ.นนทบุรี         นนทบุรี
  FLOOD จ.พระนครศรีอยุธยา พระนครศรีอยุธยา
  FLOOD         จ.อ่างทอง         อ่างทอง
  FLOOD          จ.ลพบุรี          ลพบุรี
  FLOOD       จ.สิงห์บุรี       สิงห์บุรี
  FLOOD         จ.สระบุรี         สระบุรี
  FLOOD       จ.นครสวรรค์       นครสวรรค์
  FLOOD       จ.อุทัยธานี       อุทัยธานี
  FLOOD      จ.สุพรรณบุรี      สุพรรณบุรี
  FLOOD          จ.นครปฐม          นครปฐม
  FLOOD จ.ประจวบคีรีขันธ์ ประจวบคีรีขันธ์
  FLOOD   จ.นครศรีธรรมราช   นครศรีธรรมราช
  FLOOD    จ.สุราษฎร์ธานี    สุราษฎร์ธานี
  FLOOD           จ.ชุมพร           ชุมพร
  FLOOD           จ.สงขลา           สงขลา
  FLOOD            จ.ตรัง            ตรัง
  FLOOD          จ.พัทลุง          พัทลุง
  FLOOD         จ.ปัตตานี         ปัตตานี
  FLOOD            จ.ยะลา            ยะลา
  FLOOD        จ.นราธิวาส        นราธิวาส
  FLOOD     จ.สมุทรปราการ     สมุทรปราการ
  FLOOD        จ.ปทุมธานี        ปทุมธ

In [ ]:
# ─── Validate หลังแก้ไข ───
print('=' * 60)
print('  ✅ Province Name Validation (หลังแก้ไข)')
print('=' * 60)

rain_names_fixed  = set(df_rain['PROV_T'].unique())
flood_names_fixed = set(df_flood['PROV_T'].unique())

rain_still_missing  = sorted(rain_names_fixed - ref_names)
flood_still_missing = sorted(flood_names_fixed - ref_names)

print(f'\n📊 RAIN:')
print(f'  ✅ ตรงกับ Reference : {len(rain_names_fixed & ref_names)} / {len(rain_names_fixed)} จังหวัด')
if rain_still_missing:
    print(f'  ❌ ยังไม่ตรง (ต้องเพิ่ม MANUAL_MAP): {rain_still_missing}')
else:
    print(f'  ✅ ทุกจังหวัดตรงกับ Reference ครบถ้วน')

print(f'\n📊 FLOOD:')
print(f'  ✅ ตรงกับ Reference : {len(flood_names_fixed & ref_names)} / {len(flood_names_fixed)} จังหวัด')
if flood_still_missing:
    print(f'  ❌ ยังไม่ตรง (ต้องเพิ่ม MANUAL_MAP): {flood_still_missing}')
else:
    print(f'  ✅ ทุกจังหวัดตรงกับ Reference ครบถ้วน')

# ── Merge Province EN + ภาค จาก Reference ──
df_rain = df_rain.merge(
    df_prov_ref.rename(columns={'จังหวัด': 'PROV_T', 'Province': 'PROV_EN', 'ภาค': 'ภาค_REF'}),
    on='PROV_T', how='left'
)
df_flood = df_flood.merge(
    df_prov_ref.rename(columns={'จังหวัด': 'PROV_T', 'Province': 'PROV_EN', 'ภาค': 'ภาค_REF'}),
    on='PROV_T', how='left'
)

null_rain  = df_rain['PROV_EN'].isnull().sum()
null_flood = df_flood['PROV_EN'].isnull().sum()

print(f'\n✅ Merge PROV_EN และ ภาค_REF จาก Reference สำเร็จ')
print(f'   RAIN  — PROV_EN null: {null_rain} แถว')
print(f'   FLOOD — PROV_EN null: {null_flood} แถว')

if null_rain == 0 and null_flood == 0:
    print('\n🎉 ทุกจังหวัดใน dataset ถูก Normalize และ Merge กับ Reference สำเร็จ 100%')
else:
    print('\n⚠️ ยังมีจังหวัดที่ไม่ตรงกับ Reference — กรุณาตรวจสอบ MANUAL_MAP')

print('\n💡 Interpretation:')
print('   • PROV_T_orig เก็บชื่อดั้งเดิมไว้ครบสำหรับ Traceability')
print('   • PROV_EN และ ภาค_REF พร้อมใช้งานใน EDA และ Looker Studio')


  ✅ Province Name Validation (หลังแก้ไข)

📊 RAIN:
  ✅ ตรงกับ Reference : 77 / 77 จังหวัด
  ✅ ทุกจังหวัดตรงกับ Reference ครบถ้วน

📊 FLOOD:
  ✅ ตรงกับ Reference : 75 / 75 จังหวัด
  ✅ ทุกจังหวัดตรงกับ Reference ครบถ้วน

✅ Merge PROV_EN และ ภาค_REF จาก Reference สำเร็จ
   RAIN  — PROV_EN null: 0 แถว
   FLOOD — PROV_EN null: 0 แถว

🎉 ทุกจังหวัดใน dataset ถูก Normalize และ Merge กับ Reference สำเร็จ 100%

💡 Interpretation:
   • PROV_T_orig เก็บชื่อดั้งเดิมไว้ครบสำหรับ Traceability
   • PROV_EN และ ภาค_REF พร้อมใช้งานใน EDA และ Looker Studio


In [ ]:
# ─── เปรียบเทียบ Schema ───
def schema_summary(df, name):
    info = pd.DataFrame({
        'คอลัมน์': df.columns,
        'ประเภทข้อมูล': df.dtypes.values,
        'Non-Null': df.notnull().sum().values,
        '% Non-Null': (df.notnull().mean() * 100).round(2).values,
        'ตัวอย่างค่า': [str(df[c].dropna().iloc[0]) if df[c].notnull().any() else 'NaN' for c in df.columns]
    })
    print(f'\n{'='*60}')
    print(f'  Schema: {name}')
    print(f'{'='*60}')
    print(info.to_string(index=False))
    return info

schema_rain  = schema_summary(df_rain,  'spatial-rain-hii.csv')
schema_flood = schema_summary(df_flood, 'monthly-flood-risk-area.csv')


  Schema: spatial-rain-hii.csv
    คอลัมน์ ประเภทข้อมูล  Non-Null  % Non-Null   ตัวอย่างค่า
       YEAR        int64      7546       100.0          2018
      MONTH        int64      7546       100.0             1
    PROV_ID        int64      7546       100.0            10
     PROV_T       object      7546       100.0 กรุงเทพมหานคร
    MinRain      float64      7546       100.0   54.29999924
    MaxRain      float64      7546       100.0    257.230011
    AvgRain      float64      7546       100.0   142.1191368
PROV_T_orig       object      7546       100.0 กรุงเทพมหานคร
    PROV_EN       object      7546       100.0       Bangkok
    ภาค_REF       object      7546       100.0          กลาง

  Schema: monthly-flood-risk-area.csv
      คอลัมน์ ประเภทข้อมูล  Non-Null  % Non-Null          ตัวอย่างค่า
        Month        int64     27023       100.0                    1
      GEOCODE        int64     27023       100.0               120501
     TAMBON_T       object     27023       100.0

---
# 2️⃣ Data Quality Audit

### ตรวจสอบคุณภาพข้อมูลอย่างละเอียด

ตรวจหา: **Null values**, **Duplicate rows**, **Garbage/Outlier data**  
บันทึกทุกสิ่งที่พบใน **Audit Trail**


In [ ]:
audit_trail = []  # บันทึก audit log

def log_audit(dataset, check, detail, count, severity='INFO'):
    icon = {'INFO': 'ℹ️', 'WARN': '⚠️', 'ERROR': '❌', 'OK': '✅'}.get(severity, 'ℹ️')
    entry = {'Dataset': dataset, 'Check': check, 'Detail': detail, 'Count': count, 'Severity': severity}
    audit_trail.append(entry)
    print(f'{icon} [{dataset}] {check}: {detail} → {count}')

print('▶ เริ่ม Data Quality Audit\n')

# ─── 1. Null Values ───
for name, df in [('RAIN', df_rain), ('FLOOD', df_flood)]:
    null_counts = df.isnull().sum()
    for col, cnt in null_counts.items():
        sev = 'OK' if cnt == 0 else 'WARN' if cnt < 100 else 'ERROR'
        log_audit(name, 'NULL', col, cnt, sev)

print()

# ─── 2. Duplicate Rows ───
rain_dup  = df_rain.duplicated().sum()
flood_dup = df_flood.duplicated().sum()
log_audit('RAIN',  'DUPLICATE', 'แถวซ้ำทั้งหมด', rain_dup,  'OK' if rain_dup == 0 else 'WARN')
log_audit('FLOOD', 'DUPLICATE', 'แถวซ้ำทั้งหมด', flood_dup, 'OK' if flood_dup == 0 else 'WARN')

# ─── Duplicate Key Check ───
rain_key_dup  = df_rain.duplicated(subset=['YEAR','MONTH','PROV_ID']).sum()
flood_key_dup = df_flood.duplicated(subset=['Month','GEOCODE']).sum()
log_audit('RAIN',  'DUPLICATE_KEY', 'YEAR+MONTH+PROV_ID', rain_key_dup,  'OK' if rain_key_dup == 0 else 'ERROR')
log_audit('FLOOD', 'DUPLICATE_KEY', 'Month+GEOCODE',      flood_key_dup, 'OK' if flood_key_dup == 0 else 'ERROR')

print()

# ─── 3. Garbage / Outlier Check ───
# ค่าฝนติดลบ
neg_rain = (df_rain[['MinRain','MaxRain','AvgRain']] < 0).sum().sum()
log_audit('RAIN', 'GARBAGE', 'ค่าฝนติดลบ (ไม่ควรมี)', neg_rain, 'OK' if neg_rain == 0 else 'ERROR')

# MinRain > MaxRain
min_gt_max = (df_rain['MinRain'] > df_rain['MaxRain']).sum()
log_audit('RAIN', 'LOGIC_ERROR', 'MinRain > MaxRain', min_gt_max, 'OK' if min_gt_max == 0 else 'ERROR')

# AvgRain นอกช่วง Min-Max
avg_out = ((df_rain['AvgRain'] < df_rain['MinRain']) | (df_rain['AvgRain'] > df_rain['MaxRain'])).sum()
log_audit('RAIN', 'LOGIC_ERROR', 'AvgRain นอกช่วง Min-Max', avg_out, 'OK' if avg_out == 0 else 'WARN')

# ค่า MaxRain สูงผิดปกติ (> 1000 mm/เดือน ถือว่า extreme)
extreme_rain = (df_rain['MaxRain'] > 1000).sum()
log_audit('RAIN', 'OUTLIER', 'MaxRain > 1000 mm', extreme_rain, 'WARN' if extreme_rain > 0 else 'OK')

# RISK ค่าที่ไม่อยู่ใน domain
valid_risk = ['เสี่ยงต่ำ', 'เสี่ยงปานกลาง', 'เสี่ยงสูง']
invalid_risk = (~df_flood['RISK'].isin(valid_risk)).sum()
log_audit('FLOOD', 'GARBAGE', 'RISK ค่าไม่ถูกต้อง', invalid_risk, 'OK' if invalid_risk == 0 else 'ERROR')

# COUNT 17 YEAR ค่าลบ
count_col = 'COUNT 17 YEAR'
neg_count = (df_flood[count_col] < 0).sum()
log_audit('FLOOD', 'GARBAGE', 'COUNT 17 YEAR ติดลบ', neg_count, 'OK' if neg_count == 0 else 'ERROR')

# COUNT > 17 (เกินรอบ 17 ปี)
over17 = (df_flood[count_col] > 17).sum()
log_audit('FLOOD', 'GARBAGE', 'COUNT 17 YEAR > 17', over17, 'WARN' if over17 > 0 else 'OK')

print('\n✅ Audit เสร็จสมบูรณ์')

▶ เริ่ม Data Quality Audit

✅ [RAIN] NULL: YEAR → 0
✅ [RAIN] NULL: MONTH → 0
✅ [RAIN] NULL: PROV_ID → 0
✅ [RAIN] NULL: PROV_T → 0
✅ [RAIN] NULL: MinRain → 0
✅ [RAIN] NULL: MaxRain → 0
✅ [RAIN] NULL: AvgRain → 0
✅ [RAIN] NULL: PROV_T_orig → 0
✅ [RAIN] NULL: PROV_EN → 0
✅ [RAIN] NULL: ภาค_REF → 0
✅ [FLOOD] NULL: Month → 0
✅ [FLOOD] NULL: GEOCODE → 0
✅ [FLOOD] NULL: TAMBON_T → 0
✅ [FLOOD] NULL: TAMBON_E → 0
✅ [FLOOD] NULL: AMPHOE_CODE → 0
✅ [FLOOD] NULL: AMPHOE_T → 0
✅ [FLOOD] NULL: AMPHOE_E → 0
✅ [FLOOD] NULL: PROV_CODE → 0
✅ [FLOOD] NULL: PROV_T → 0
✅ [FLOOD] NULL: PROV_E → 0
✅ [FLOOD] NULL: COUNT 17 YEAR → 0
✅ [FLOOD] NULL: CRITERIA → 0
✅ [FLOOD] NULL: RISK → 0
✅ [FLOOD] NULL: PROV_T_orig → 0
✅ [FLOOD] NULL: PROV_EN → 0
✅ [FLOOD] NULL: ภาค_REF → 0

✅ [RAIN] DUPLICATE: แถวซ้ำทั้งหมด → 0
✅ [FLOOD] DUPLICATE: แถวซ้ำทั้งหมด → 0
✅ [RAIN] DUPLICATE_KEY: YEAR+MONTH+PROV_ID → 0
✅ [FLOOD] DUPLICATE_KEY: Month+GEOCODE → 0

✅ [RAIN] GARBAGE: ค่าฝนติดลบ (ไม่ควรมี) → 0
✅ [RAIN] LOGIC_ERROR: MinRain

In [ ]:
# ─── สรุป Audit Trail ───
df_audit = pd.DataFrame(audit_trail)

print('\n📋 AUDIT TRAIL SUMMARY')
print('='*70)
print(df_audit.to_string(index=False))

print('\n📊 สรุปตาม Severity:')
print(df_audit['Severity'].value_counts().to_string())


📋 AUDIT TRAIL SUMMARY
Dataset         Check                  Detail  Count Severity
   RAIN          NULL                    YEAR      0       OK
   RAIN          NULL                   MONTH      0       OK
   RAIN          NULL                 PROV_ID      0       OK
   RAIN          NULL                  PROV_T      0       OK
   RAIN          NULL                 MinRain      0       OK
   RAIN          NULL                 MaxRain      0       OK
   RAIN          NULL                 AvgRain      0       OK
   RAIN          NULL             PROV_T_orig      0       OK
   RAIN          NULL                 PROV_EN      0       OK
   RAIN          NULL                 ภาค_REF      0       OK
  FLOOD          NULL                   Month      0       OK
  FLOOD          NULL                 GEOCODE      0       OK
  FLOOD          NULL                TAMBON_T      0       OK
  FLOOD          NULL                TAMBON_E      0       OK
  FLOOD          NULL             AMPHOE_CODE  

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Null Values: RAIN', 'Null Values: FLOOD')
)

for col_idx, (df, name) in enumerate([(df_rain, 'RAIN'), (df_flood, 'FLOOD')], start=1):
    null_pct = (df.isnull().mean() * 100).reset_index()
    null_pct.columns = ['คอลัมน์', '% Null']
    null_pct['% Null'] = null_pct['% Null'].round(2)
    null_pct['สี'] = null_pct['% Null'].apply(lambda v: '#E53935' if v > 0 else '#43A047')
    null_pct['label'] = null_pct['% Null'].apply(
        lambda v: f'{v:.2f}%' if v > 0 else '✅ ไม่มี Null'
    )

    fig.add_trace(
        go.Bar(
            x=null_pct['% Null'],
            y=null_pct['คอลัมน์'],
            orientation='h',
            marker_color=null_pct['สี'],
            text=null_pct['label'],
            textposition='outside',
            hovertemplate='%{y}: %{x:.2f}%<extra></extra>',
            name=name,
            showlegend=False,
        ),
        row=1, col=col_idx
    )

fig.update_xaxes(title_text='% Null', range=[-0.05, 5])
fig.update_layout(
    title=dict(text='🔍 การตรวจสอบ Null Values', x=0.5),
    height=max(350, max(len(df_rain.columns), len(df_flood.columns)) * 22),
    template='plotly_white',
    margin=dict(l=160, r=120),
)
fig.show()

print('\n💡 Interpretation:')
print('   • สีเขียว = ไม่มี Null ✅  |  สีแดง = มี Null ❌')
print('   • แกน x เริ่มที่ 0 เสมอ ทำให้เห็นได้ว่าคอลัมน์ไหนสะอาด')


💡 Interpretation:
   • สีเขียว = ไม่มี Null ✅  |  สีแดง = มี Null ❌
   • แกน x เริ่มที่ 0 เสมอ ทำให้เห็นได้ว่าคอลัมน์ไหนสะอาด


---
# 3️⃣ Data Cleaning

### ทำความสะอาดข้อมูลตามที่พบใน Audit

- แก้ไข Type ของคอลัมน์ให้ถูกต้อง
- กรอง Noise ด้วย Threshold ที่เหมาะสม (AvgRain > 0 สำหรับการวิเคราะห์หลัก)
- สร้าง YEAR_MONTH label สำหรับ Time Series
- Log การแก้ไขทุกขั้นตอน


In [ ]:
clean_log = []  # บันทึก cleaning log

def log_clean(step, detail, before, after):
    diff = before - after
    clean_log.append({'ขั้นตอน': step, 'รายละเอียด': detail, 'ก่อน': before, 'หลัง': after, 'ลบออก': diff})
    print(f'🧹 {step}: {detail} | ก่อน={before:,} → หลัง={after:,} | ลบ={diff:,}')

# ─── Copy เพื่อทำงาน ───
df_r = df_rain.copy()
df_f = df_flood.copy()
print(f'ขนาดเริ่มต้น — RAIN: {len(df_r):,}  FLOOD: {len(df_f):,}\n')

# ─── RAIN: แก้ไข Type ───
df_r['YEAR']  = df_r['YEAR'].astype(int)
df_r['MONTH'] = df_r['MONTH'].astype(int)
df_r['PROV_ID'] = df_r['PROV_ID'].astype(str).str.zfill(2)
print('✅ RAIN — แปลง YEAR/MONTH เป็น int, PROV_ID เป็น string 2 หลัก')

# ─── RAIN: ลบ Duplicate ───
before = len(df_r)
df_r.drop_duplicates(subset=['YEAR','MONTH','PROV_ID'], keep='first', inplace=True)
log_clean('RAIN', 'ลบ duplicate key (YEAR+MONTH+PROV_ID)', before, len(df_r))

# ─── RAIN: กรอง Noise — AvgRain >= 0 (ค่าติดลบไม่มีความหมาย) ───
before = len(df_r)
df_r = df_r[df_r['AvgRain'] >= 0].copy()
log_clean('RAIN', 'กรอง AvgRain < 0 (noise)', before, len(df_r))

# ─── RAIN: สร้าง Feature ใหม่ ───
month_th = {1:'ม.ค.',2:'ก.พ.',3:'มี.ค.',4:'เม.ย.',5:'พ.ค.',6:'มิ.ย.',
            7:'ก.ค.',8:'ส.ค.',9:'ก.ย.',10:'ต.ค.',11:'พ.ย.',12:'ธ.ค.'}
df_r['MONTH_TH'] = df_r['MONTH'].map(month_th)
df_r['YEAR_MONTH'] = df_r['YEAR'].astype(str) + '-' + df_r['MONTH'].astype(str).str.zfill(2)
df_r['RAIN_RANGE'] = df_r['MaxRain'] - df_r['MinRain']
df_r['DATE'] = pd.to_datetime(df_r['YEAR'].astype(str) + '-' + df_r['MONTH'].astype(str).str.zfill(2) + '-01')
print('✅ RAIN — สร้าง MONTH_TH, YEAR_MONTH, RAIN_RANGE, DATE')

print()

# ─── FLOOD: แก้ไข Type ───
df_f['GEOCODE']   = df_f['GEOCODE'].astype(str).str.zfill(6)
df_f['PROV_CODE'] = df_f['PROV_CODE'].astype(str).str.zfill(2)
df_f['AMPHOE_CODE'] = df_f['AMPHOE_CODE'].astype(str).str.zfill(4)
df_f['Month']     = df_f['Month'].astype(int)
df_f['COUNT 17 YEAR'] = df_f['COUNT 17 YEAR'].astype(int)
print('✅ FLOOD — แปลง GEOCODE/PROV_CODE/AMPHOE_CODE เป็น string, Month เป็น int')

# ─── FLOOD: ลบ Duplicate ───
before = len(df_f)
df_f.drop_duplicates(subset=['Month','GEOCODE'], keep='first', inplace=True)
log_clean('FLOOD', 'ลบ duplicate key (Month+GEOCODE)', before, len(df_f))

# ─── FLOOD: กรอง RISK ที่ไม่ถูกต้อง ───
before = len(df_f)
df_f = df_f[df_f['RISK'].isin(['เสี่ยงต่ำ','เสี่ยงปานกลาง','เสี่ยงสูง'])].copy()
log_clean('FLOOD', 'กรอง RISK ค่าไม่ถูกต้อง', before, len(df_f))

# ─── FLOOD: สร้าง Feature ใหม่ ───
df_f['MONTH_TH'] = df_f['Month'].map(month_th)
risk_order = {'เสี่ยงต่ำ': 1, 'เสี่ยงปานกลาง': 2, 'เสี่ยงสูง': 3}
df_f['RISK_SCORE'] = df_f['RISK'].map(risk_order)
print('✅ FLOOD — สร้าง MONTH_TH, RISK_SCORE')

print(f'\n📦 ขนาดหลัง Cleaning — RAIN: {len(df_r):,}  FLOOD: {len(df_f):,}')

ขนาดเริ่มต้น — RAIN: 7,546  FLOOD: 27,023

✅ RAIN — แปลง YEAR/MONTH เป็น int, PROV_ID เป็น string 2 หลัก
🧹 RAIN: ลบ duplicate key (YEAR+MONTH+PROV_ID) | ก่อน=7,546 → หลัง=7,546 | ลบ=0
🧹 RAIN: กรอง AvgRain < 0 (noise) | ก่อน=7,546 → หลัง=7,546 | ลบ=0
✅ RAIN — สร้าง MONTH_TH, YEAR_MONTH, RAIN_RANGE, DATE

✅ FLOOD — แปลง GEOCODE/PROV_CODE/AMPHOE_CODE เป็น string, Month เป็น int
🧹 FLOOD: ลบ duplicate key (Month+GEOCODE) | ก่อน=27,023 → หลัง=27,023 | ลบ=0
🧹 FLOOD: กรอง RISK ค่าไม่ถูกต้อง | ก่อน=27,023 → หลัง=27,023 | ลบ=0
✅ FLOOD — สร้าง MONTH_TH, RISK_SCORE

📦 ขนาดหลัง Cleaning — RAIN: 7,546  FLOOD: 27,023


In [ ]:
# ─── Cleaning Log Summary ───
df_clean_log = pd.DataFrame(clean_log)
print('\n📋 CLEANING LOG')
print('='*70)
print(df_clean_log.to_string(index=False))

print('\n✅ Data Cleaning เสร็จสมบูรณ์ ข้อมูลพร้อมใช้งาน')


📋 CLEANING LOG
ขั้นตอน                            รายละเอียด  ก่อน  หลัง  ลบออก
   RAIN ลบ duplicate key (YEAR+MONTH+PROV_ID)  7546  7546      0
   RAIN              กรอง AvgRain < 0 (noise)  7546  7546      0
  FLOOD      ลบ duplicate key (Month+GEOCODE) 27023 27023      0
  FLOOD               กรอง RISK ค่าไม่ถูกต้อง 27023 27023      0

✅ Data Cleaning เสร็จสมบูรณ์ ข้อมูลพร้อมใช้งาน


---
# 4️⃣ EDA เบื้องต้น

### การวิเคราะห์เชิงสถิติเบื้องต้น

- Distribution ของตัวแปรหลัก
- KPI สำคัญที่ต้องรู้


In [ ]:
# ─── KPI Summary ───
print('='*60)
print('  📊 KPI SUMMARY — ข้อมูลปริมาณฝน')
print('='*60)
print(f"  🗓️  ช่วงเวลา         : {df_r['YEAR'].min()}–{df_r['YEAR'].max()} ({df_r['YEAR'].nunique()} ปี)")
print(f"  🗺️  จำนวนจังหวัด    : {df_r['PROV_T'].nunique()} จังหวัด")
print(f"  🌧️  AvgRain เฉลี่ย  : {df_r['AvgRain'].mean():.1f} mm")
print(f"  🌧️  AvgRain สูงสุด  : {df_r['AvgRain'].max():.1f} mm ({df_r.loc[df_r['AvgRain'].idxmax(), 'PROV_T']})")
print(f"  🌧️  MaxRain สูงสุด  : {df_r['MaxRain'].max():.1f} mm")
print(f"  📈  จำนวนระเบียน    : {len(df_r):,} แถว")

print()
print('='*60)
print('  📊 KPI SUMMARY — ข้อมูลพื้นที่เสี่ยงน้ำท่วม')
print('='*60)
total_tambon = df_f['GEOCODE'].nunique()
high_risk    = df_f[df_f['RISK']=='เสี่ยงสูง']['GEOCODE'].nunique()
mid_risk     = df_f[df_f['RISK']=='เสี่ยงปานกลาง']['GEOCODE'].nunique()
low_risk     = df_f[df_f['RISK']=='เสี่ยงต่ำ']['GEOCODE'].nunique()
print(f"  🗺️  จำนวนตำบล (unique)    : {total_tambon:,}")
print(f"  ❗ เสี่ยงสูง               : {high_risk:,} ตำบล")
print(f"  ⚠️  เสี่ยงปานกลาง         : {mid_risk:,} ตำบล")
print(f"  ✅ เสี่ยงต่ำ               : {low_risk:,} ตำบล")
print(f"  🏙️  จำนวนจังหวัด (unique) : {df_f['PROV_CODE'].nunique()}")
print(f"  📈  จำนวนระเบียน           : {len(df_f):,} แถว")

  📊 KPI SUMMARY — ข้อมูลปริมาณฝน
  🗓️  ช่วงเวลา         : 2018–2026 (9 ปี)
  🗺️  จำนวนจังหวัด    : 77 จังหวัด
  🌧️  AvgRain เฉลี่ย  : 130.0 mm
  🌧️  AvgRain สูงสุด  : 1519.6 mm (นราธิวาส)
  🌧️  MaxRain สูงสุด  : 1647.8 mm
  📈  จำนวนระเบียน    : 7,546 แถว

  📊 KPI SUMMARY — ข้อมูลพื้นที่เสี่ยงน้ำท่วม
  🗺️  จำนวนตำบล (unique)    : 6,435
  ❗ เสี่ยงสูง               : 1,445 ตำบล
  ⚠️  เสี่ยงปานกลาง         : 3,787 ตำบล
  ✅ เสี่ยงต่ำ               : 6,402 ตำบล
  🏙️  จำนวนจังหวัด (unique) : 75
  📈  จำนวนระเบียน           : 27,023 แถว


In [ ]:
# ─── Distribution of AvgRain ───
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Distribution ของ AvgRain (mm)', 'Boxplot AvgRain รายปี'))

# Histogram
fig.add_trace(
    go.Histogram(x=df_r['AvgRain'], nbinsx=60,
                 marker_color=COLOR_BLUE, opacity=0.75, name='AvgRain'),
    row=1, col=1)

# Boxplot per year
for yr in sorted(df_r['YEAR'].unique()):
    sub = df_r[df_r['YEAR']==yr]['AvgRain']
    fig.add_trace(
        go.Box(y=sub, name=str(yr), showlegend=False,
               marker_color=COLOR_BLUE, line_color=COLOR_BLUE),
        row=1, col=2)

fig.update_layout(
    title='📊 การกระจายตัวของปริมาณฝนเฉลี่ย (AvgRain)',
    height=450, template='plotly_white',
    xaxis_title='AvgRain (mm)', yaxis_title='ความถี่',
    yaxis2_title='AvgRain (mm)'
)
fig.show()

print("""
💡 Interpretation:
   • การกระจายของ AvgRain มีลักษณะ Right-Skewed — ส่วนใหญ่ต่ำกว่า 200 mm/เดือน
   • Boxplot รายปีช่วยเห็นความผันผวนในแต่ละปี
   • ปีที่มี Median สูงกว่าปกติ มักสัมพันธ์กับช่วงมรสุมที่รุนแรง
""")


💡 Interpretation:
   • การกระจายของ AvgRain มีลักษณะ Right-Skewed — ส่วนใหญ่ต่ำกว่า 200 mm/เดือน
   • Boxplot รายปีช่วยเห็นความผันผวนในแต่ละปี
   • ปีที่มี Median สูงกว่าปกติ มักสัมพันธ์กับช่วงมรสุมที่รุนแรง



In [ ]:
# ─── Distribution ของระดับความเสี่ยงน้ำท่วม ───
risk_dist = df_f.groupby(['RISK','RISK_SCORE'])['GEOCODE'].nunique().reset_index()
risk_dist = risk_dist.sort_values('RISK_SCORE')
risk_dist.columns = ['RISK','RISK_SCORE','จำนวนตำบล']

fig = px.bar(risk_dist, x='RISK', y='จำนวนตำบล',
             color='RISK', color_discrete_map=RISK_COLORS,
             title='📊 จำนวนตำบลแยกตามระดับความเสี่ยงน้ำท่วม',
             text='จำนวนตำบล', template='plotly_white')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='ระดับความเสี่ยง', yaxis_title='จำนวนตำบล (unique)')
fig.show()

print("""
💡 Interpretation:
   • ตำบลส่วนใหญ่อยู่ในระดับเสี่ยงต่ำ แต่ตำบลเสี่ยงสูงก็มีจำนวนมีนัยสำคัญ
   • ข้อมูล COUNT 17 YEAR สะท้อนความถี่น้ำท่วมสะสมย้อนหลัง 17 ปี
   • ตำบลเสี่ยงสูง = ท่วมซ้ำ 9–17 ครั้งใน 17 ปี (เกือบทุกปี)
""")


💡 Interpretation:
   • ตำบลส่วนใหญ่อยู่ในระดับเสี่ยงต่ำ แต่ตำบลเสี่ยงสูงก็มีจำนวนมีนัยสำคัญ
   • ข้อมูล COUNT 17 YEAR สะท้อนความถี่น้ำท่วมสะสมย้อนหลัง 17 ปี
   • ตำบลเสี่ยงสูง = ท่วมซ้ำ 9–17 ครั้งใน 17 ปี (เกือบทุกปี)



In [ ]:
# ─── Stats Summary Table ───
print('📋 สถิติเชิงพรรณนา — RAIN (AvgRain, MinRain, MaxRain)')
display(df_r[['MinRain','MaxRain','AvgRain','RAIN_RANGE']].describe().round(2))

print('\n📋 สถิติเชิงพรรณนา — FLOOD (COUNT 17 YEAR, RISK_SCORE)')
display(df_f[['COUNT 17 YEAR','RISK_SCORE']].describe().round(2))

📋 สถิติเชิงพรรณนา — RAIN (AvgRain, MinRain, MaxRain)


,MinRain,MaxRain,AvgRain,RAIN_RANGE
count,7546.00,7546.00,7546.00,7546.00
mean,85.71,189.20,130.01,103.49
std,103.10,192.67,136.03,118.28
min,0.00,0.00,0.00,0.00
25%,4.38,34.09,17.84,23.06
50%,55.41,148.16,100.44,69.68
75%,134.72,277.73,197.85,139.71
max,1421.49,1647.78,1519.62,956.32



📋 สถิติเชิงพรรณนา — FLOOD (COUNT 17 YEAR, RISK_SCORE)


,COUNT 17 YEAR,RISK_SCORE
count,27023.00,27023.00
mean,3.29,1.41
std,2.98,0.64
min,1.00,1.00
25%,1.00,1.00
50%,2.00,1.00
75%,5.00,2.00
max,14.00,3.00


---
# 5️⃣ Advanced EDA & Insights

### วิเคราะห์เชิงลึกหลายมิติ

- **Time Series**: แนวโน้มปริมาณฝนตลอดปี 2018–2026
- **Seasonal Pattern**: รูปแบบฝนรายเดือนและรายฤดูกาล
- **Geographic**: จังหวัดฝนมาก–น้อย และพื้นที่เสี่ยงสูง
- **Risk Analysis**: ความสัมพันธ์ฝน × ความเสี่ยงน้ำท่วม
- **Regional Comparison**: เปรียบเทียบภาค


In [ ]:
# ─── 5.1 Time Series: แนวโน้มปริมาณฝนเฉลี่ยทั้งประเทศ ───
ts = df_r.groupby('DATE')['AvgRain'].mean().reset_index()
ts_year = df_r.groupby('YEAR')['AvgRain'].mean().reset_index()

fig = make_subplots(rows=2, cols=1,
    subplot_titles=('ปริมาณฝนเฉลี่ยรายเดือน (ทั้งประเทศ)', 'ปริมาณฝนเฉลี่ยรายปี'),
    vertical_spacing=0.12)

# Monthly time series
fig.add_trace(
    go.Scatter(x=ts['DATE'], y=ts['AvgRain'],
               mode='lines', line=dict(color=COLOR_BLUE, width=1.5),
               name='AvgRain รายเดือน', fill='tozeroy', fillcolor='rgba(30,136,229,0.1)'),
    row=1, col=1)

# Yearly bar
fig.add_trace(
    go.Bar(x=ts_year['YEAR'], y=ts_year['AvgRain'],
           marker_color=[COLOR_RED if v > ts_year['AvgRain'].mean() else COLOR_BLUE for v in ts_year['AvgRain']],
           name='AvgRain รายปี', text=ts_year['AvgRain'].round(1),
           textposition='outside'),
    row=2, col=1)

# Mean line
mean_val = ts_year['AvgRain'].mean()
fig.add_hline(y=mean_val, line_dash='dash', line_color='gray',
              annotation_text=f'ค่าเฉลี่ย {mean_val:.1f} mm', row=2, col=1)

fig.update_layout(
    title='📈 แนวโน้มปริมาณฝน 2018–2026',
    height=600, template='plotly_white', showlegend=False
)
fig.update_xaxes(title_text='เดือน', row=1, col=1)
fig.update_xaxes(title_text='ปี', row=2, col=1)
fig.update_yaxes(title_text='AvgRain (mm)', row=1, col=1)
fig.update_yaxes(title_text='AvgRain (mm)', row=2, col=1)
fig.show()

# หา peak years
peak_year = ts_year.loc[ts_year['AvgRain'].idxmax(), 'YEAR']
low_year  = ts_year.loc[ts_year['AvgRain'].idxmin(), 'YEAR']
print(f"""
💡 Interpretation:
   • ปีที่ฝนมากที่สุด: {peak_year} ({ts_year[ts_year['YEAR']==peak_year]['AvgRain'].values[0]:.1f} mm/เดือนเฉลี่ย)
   • ปีที่ฝนน้อยที่สุด: {low_year} ({ts_year[ts_year['YEAR']==low_year]['AvgRain'].values[0]:.1f} mm/เดือนเฉลี่ย)
   • แถบสีแดง = ปีที่ฝนเกินค่าเฉลี่ย → ความเสี่ยงน้ำท่วมสูงขึ้น
   • รูปแบบ Seasonal ชัดเจน: ฝนสูงสุดช่วงกลางปี (มิ.ย.–ต.ค.)
""")


💡 Interpretation:
   • ปีที่ฝนมากที่สุด: 2022 (163.1 mm/เดือนเฉลี่ย)
   • ปีที่ฝนน้อยที่สุด: 2026 (26.8 mm/เดือนเฉลี่ย)
   • แถบสีแดง = ปีที่ฝนเกินค่าเฉลี่ย → ความเสี่ยงน้ำท่วมสูงขึ้น
   • รูปแบบ Seasonal ชัดเจน: ฝนสูงสุดช่วงกลางปี (มิ.ย.–ต.ค.)



In [ ]:
# ─── 5.2 Seasonal Pattern: Heatmap ฝนรายเดือน×ปี ───
heat = df_r.groupby(['YEAR','MONTH'])['AvgRain'].mean().reset_index()
heat_pivot = heat.pivot(index='YEAR', columns='MONTH', values='AvgRain')
heat_pivot.columns = [month_th[m] for m in heat_pivot.columns]

fig = px.imshow(
    heat_pivot,
    color_continuous_scale='Blues',
    title='🗓️ Heatmap ปริมาณฝนเฉลี่ย (mm) แยกตามปีและเดือน',
    labels={'color': 'AvgRain (mm)', 'x': 'เดือน', 'y': 'ปี'},
    template='plotly_white',
    text_auto='.0f'
)
fig.update_layout(height=400)
fig.show()

# Seasonal avg
seasonal = df_r.groupby('MONTH')['AvgRain'].mean()
peak_month = seasonal.idxmax()
low_month  = seasonal.idxmin()
print(f"""
💡 Interpretation:
   • เดือนที่ฝนมากที่สุดเฉลี่ย: {month_th[peak_month]} (เดือน {peak_month}) — {seasonal[peak_month]:.1f} mm
   • เดือนที่ฝนน้อยที่สุดเฉลี่ย: {month_th[low_month]} (เดือน {low_month}) — {seasonal[low_month]:.1f} mm
   • สีเข้ม = ฝนมาก, สีอ่อน = ฝนน้อย
   • ฤดูฝน (มิ.ย.–ต.ค.) ชัดเจนทุกปี โดยเฉพาะภาคตะวันออกและใต้
""")


💡 Interpretation:
   • เดือนที่ฝนมากที่สุดเฉลี่ย: ก.ย. (เดือน 9) — 283.1 mm
   • เดือนที่ฝนน้อยที่สุดเฉลี่ย: ม.ค. (เดือน 1) — 22.7 mm
   • สีเข้ม = ฝนมาก, สีอ่อน = ฝนน้อย
   • ฤดูฝน (มิ.ย.–ต.ค.) ชัดเจนทุกปี โดยเฉพาะภาคตะวันออกและใต้



In [ ]:
# ─── 5.3 Seasonal Bar Chart: ค่าเฉลี่ยฝนรายเดือน ───
seasonal_df = df_r.groupby(['MONTH','MONTH_TH'])['AvgRain'].agg(['mean','std']).reset_index()
seasonal_df.columns = ['MONTH','MONTH_TH','mean','std']
seasonal_df = seasonal_df.sort_values('MONTH')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=seasonal_df['MONTH_TH'], y=seasonal_df['mean'],
    error_y=dict(type='data', array=seasonal_df['std'], visible=True),
    marker_color=[COLOR_RED if v > seasonal_df['mean'].mean() else COLOR_BLUE for v in seasonal_df['mean']],
    text=seasonal_df['mean'].round(1), textposition='outside',
    name='AvgRain เฉลี่ย'
))
fig.update_layout(
    title='🌦️ ปริมาณฝนเฉลี่ยรายเดือน (เฉลี่ยทุกปีและทุกจังหวัด)',
    xaxis_title='เดือน', yaxis_title='AvgRain เฉลี่ย (mm)',
    template='plotly_white', height=420, showlegend=False
)
fig.show()

print("""
💡 Interpretation:
   • Error bar แสดงความผันผวน (Std Dev) — เดือนฝนมาก มักมี Std Dev สูง
   • สีแดง = เดือนที่ฝนเกินค่าเฉลี่ยรายปี
   • ช่วง ก.ค.–ต.ค. คือช่วงที่ปริมาณฝนสูงและผันผวนมากที่สุด
""")


💡 Interpretation:
   • Error bar แสดงความผันผวน (Std Dev) — เดือนฝนมาก มักมี Std Dev สูง
   • สีแดง = เดือนที่ฝนเกินค่าเฉลี่ยรายปี
   • ช่วง ก.ค.–ต.ค. คือช่วงที่ปริมาณฝนสูงและผันผวนมากที่สุด



In [ ]:
# ─── 5.4 Geographic: Top จังหวัดฝนมาก–น้อย ───
prov_avg = df_r.groupby('PROV_T')['AvgRain'].mean().reset_index()
prov_avg.columns = ['จังหวัด','AvgRain เฉลี่ย (mm)']
prov_avg = prov_avg.sort_values('AvgRain เฉลี่ย (mm)', ascending=False)

top10 = prov_avg.head(10)
bot10 = prov_avg.tail(10)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('🌧️ Top 10 จังหวัดฝนมากที่สุด', '☀️ Top 10 จังหวัดฝนน้อยที่สุด'))

fig.add_trace(go.Bar(x=top10['AvgRain เฉลี่ย (mm)'], y=top10['จังหวัด'],
                     orientation='h', marker_color=COLOR_BLUE, name='ฝนมาก',
                     text=top10['AvgRain เฉลี่ย (mm)'].round(1), textposition='outside'),
              row=1, col=1)

fig.add_trace(go.Bar(x=bot10['AvgRain เฉลี่ย (mm)'], y=bot10['จังหวัด'],
                     orientation='h', marker_color=COLOR_ORANGE, name='ฝนน้อย',
                     text=bot10['AvgRain เฉลี่ย (mm)'].round(1), textposition='outside'),
              row=1, col=2)

fig.update_layout(
    title='🗺️ การกระจายปริมาณฝนตามจังหวัด',
    height=500, template='plotly_white', showlegend=False
)
fig.update_xaxes(title_text='AvgRain เฉลี่ย (mm)')
fig.show()

print(f"""
💡 Interpretation:
   • จังหวัดฝนมากที่สุด: {top10.iloc[0]['จังหวัด']} ({top10.iloc[0]['AvgRain เฉลี่ย (mm)']:.1f} mm)
   • จังหวัดฝนน้อยที่สุด: {bot10.iloc[-1]['จังหวัด']} ({bot10.iloc[-1]['AvgRain เฉลี่ย (mm)']:.1f} mm)
   • ภาคใต้และภาคตะวันออกมักมีฝนมากกว่าภาคอื่น
   • ภาคตะวันออกเฉียงเหนือตอนบน (ใกล้ลาว) ฝนน้อยกว่าเฉลี่ย
""")


💡 Interpretation:
   • จังหวัดฝนมากที่สุด: ตราด (268.2 mm)
   • จังหวัดฝนน้อยที่สุด: สุพรรณบุรี (80.2 mm)
   • ภาคใต้และภาคตะวันออกมักมีฝนมากกว่าภาคอื่น
   • ภาคตะวันออกเฉียงเหนือตอนบน (ใกล้ลาว) ฝนน้อยกว่าเฉลี่ย



In [ ]:
# ─── 5.5 Flood Risk: จำนวนตำบลเสี่ยงสูงรายจังหวัด (Top 20) ───
high_risk_prov = (
    df_f[df_f['RISK']=='เสี่ยงสูง']
    .groupby('PROV_T')['GEOCODE'].nunique()
    .reset_index()
    .sort_values('GEOCODE', ascending=False)
    .head(20)
)
high_risk_prov.columns = ['จังหวัด','ตำบลเสี่ยงสูง']

fig = px.bar(
    high_risk_prov, x='ตำบลเสี่ยงสูง', y='จังหวัด',
    orientation='h', color='ตำบลเสี่ยงสูง',
    color_continuous_scale='Reds',
    title='❗ Top 20 จังหวัดที่มีตำบลเสี่ยงน้ำท่วมสูงมากที่สุด',
    text='ตำบลเสี่ยงสูง', template='plotly_white'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=550, coloraxis_showscale=False,
                  xaxis_title='จำนวนตำบล (unique)', yaxis={'categoryorder':'total ascending'})
fig.show()

top_prov = high_risk_prov.iloc[0]
print(f"""
💡 Interpretation:
   • จังหวัดที่มีตำบลเสี่ยงสูงมากที่สุด: {top_prov['จังหวัด']} ({top_prov['ตำบลเสี่ยงสูง']:,} ตำบล)
   • จังหวัดลุ่มน้ำเจ้าพระยาและลุ่มน้ำมูล-ชีมักติดอันดับต้น
   • ควรวางแผนระบบเตือนภัยล่วงหน้าสำหรับจังหวัดเหล่านี้เป็นลำดับแรก
""")


💡 Interpretation:
   • จังหวัดที่มีตำบลเสี่ยงสูงมากที่สุด: พระนครศรีอยุธยา (158 ตำบล)
   • จังหวัดลุ่มน้ำเจ้าพระยาและลุ่มน้ำมูล-ชีมักติดอันดับต้น
   • ควรวางแผนระบบเตือนภัยล่วงหน้าสำหรับจังหวัดเหล่านี้เป็นลำดับแรก



In [ ]:
# ─── 5.6 Risk Distribution รายเดือน (Stacked Bar) ───
monthly_risk = (
    df_f.groupby(['Month','MONTH_TH','RISK'])['GEOCODE'].nunique()
    .reset_index()
)
monthly_risk.columns = ['Month','MONTH_TH','RISK','จำนวนตำบล']
monthly_risk = monthly_risk.sort_values('Month')

fig = px.bar(
    monthly_risk, x='MONTH_TH', y='จำนวนตำบล',
    color='RISK', color_discrete_map=RISK_COLORS,
    barmode='stack',
    title='📅 จำนวนตำบลตามระดับความเสี่ยงน้ำท่วม แยกรายเดือน',
    template='plotly_white',
    category_orders={'RISK': ['เสี่ยงสูง','เสี่ยงปานกลาง','เสี่ยงต่ำ']}
)
fig.update_layout(
    height=420,
    xaxis_title='เดือน', yaxis_title='จำนวนตำบล'
)
fig.show()

high_months = (
    df_f[df_f['RISK']=='เสี่ยงสูง']
    .groupby(['Month','MONTH_TH'])['GEOCODE'].nunique()
    .reset_index()
    .sort_values('GEOCODE', ascending=False)
    .head(3)
)
print(f"""
💡 Interpretation:
   • เดือนที่มีตำบลเสี่ยงสูงมากที่สุด 3 อันดับ:
     {' → '.join([f"{r['MONTH_TH']} ({r['GEOCODE']:,} ตำบล)" for _, r in high_months.iterrows()])}
   • ความเสี่ยงสูงสุดช่วง ก.ย.–พ.ย. สอดคล้องกับฤดูน้ำหลาก
   • จำนวนตำบลเสี่ยงสูงเพิ่มขึ้นในช่วงปลายปี แล้วลดลงช่วงต้นปี
""")


💡 Interpretation:
   • เดือนที่มีตำบลเสี่ยงสูงมากที่สุด 3 อันดับ:
     ต.ค. (1,177 ตำบล) → ก.ย. (580 ตำบล) → พ.ย. (307 ตำบล)
   • ความเสี่ยงสูงสุดช่วง ก.ย.–พ.ย. สอดคล้องกับฤดูน้ำหลาก
   • จำนวนตำบลเสี่ยงสูงเพิ่มขึ้นในช่วงปลายปี แล้วลดลงช่วงต้นปี



In [ ]:
# ─── 5.7 ความสัมพันธ์: ปริมาณฝน vs ความเสี่ยงน้ำท่วม ───
# Join ด้วย PROV_CODE และ Month
rain_prov_month = (
    df_r.groupby(['PROV_ID','MONTH'])['AvgRain'].mean()
    .reset_index()
)
rain_prov_month.columns = ['PROV_ID','Month','AvgRain']

flood_prov_risk = (
    df_f.groupby(['PROV_CODE','Month'])['RISK_SCORE'].mean()
    .reset_index()
)
flood_prov_risk.columns = ['PROV_ID','Month','RISK_SCORE_avg']

merged = pd.merge(rain_prov_month, flood_prov_risk, on=['PROV_ID','Month'], how='inner')

fig = px.scatter(
    merged, x='AvgRain', y='RISK_SCORE_avg',
    color='Month', color_continuous_scale='Viridis',
    trendline='ols',
    title='🔗 ความสัมพันธ์ระหว่างปริมาณฝนเฉลี่ยและระดับความเสี่ยงน้ำท่วม',
    labels={'AvgRain': 'AvgRain เฉลี่ย (mm)', 'RISK_SCORE_avg': 'Risk Score เฉลี่ย (1=ต่ำ, 3=สูง)'},
    template='plotly_white', opacity=0.6
)
fig.update_layout(height=450)
fig.show()

corr = merged[['AvgRain','RISK_SCORE_avg']].corr().iloc[0,1]
print(f"""
💡 Interpretation:
   • Correlation (AvgRain × Risk Score) = {corr:.3f}
   • {'ความสัมพันธ์เชิงบวก: จังหวัดที่ฝนตกมากมีแนวโน้มเสี่ยงน้ำท่วมสูงกว่า' if corr > 0 else 'ความสัมพันธ์ซับซ้อน: ปริมาณฝนอาจไม่ใช่ปัจจัยเดียว'}
   • ปัจจัยอื่นที่มีผล: ระบบระบายน้ำ, ความลาดชันพื้นที่, การใช้ที่ดิน
   • จุดสีเข้ม (เดือน 9–11) มักอยู่ในโซนความเสี่ยงสูงกว่า
""")


💡 Interpretation:
   • Correlation (AvgRain × Risk Score) = 0.202
   • ความสัมพันธ์เชิงบวก: จังหวัดที่ฝนตกมากมีแนวโน้มเสี่ยงน้ำท่วมสูงกว่า
   • ปัจจัยอื่นที่มีผล: ระบบระบายน้ำ, ความลาดชันพื้นที่, การใช้ที่ดิน
   • จุดสีเข้ม (เดือน 9–11) มักอยู่ในโซนความเสี่ยงสูงกว่า



In [ ]:
# ─── 5.8 Regional Analysis: กลุ่มจังหวัดตามภาค ───
# แผนที่จังหวัด → ภาค (ตาม PROV_ID range)
def assign_region(prov_id):
    pid = int(prov_id)
    if pid in [10,11,12,13,14,15,16,17,18,19,24,25,26,27,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49]:
        if pid in [30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49]:
            return 'ภาคตะวันออกเฉียงเหนือ'
        elif pid in [10,11,12,13,14,15,16,17,18,19]:
            return 'ภาคกลาง'
        else:
            return 'ภาคตะวันออก'
    elif pid in [50,51,52,53,54,55,56,57,58,60,61,62,63,64,65,66,67]:
        if pid in [50,51,52,53,54,55,56,57,58]:
            return 'ภาคเหนือ'
        else:
            return 'ภาคเหนือ (ตอนล่าง)'
    elif pid in [70,71,72,73,74,75,76,77]:
        return 'ภาคตะวันตก'
    elif pid >= 80:
        return 'ภาคใต้'
    else:
        return 'ภาคกลาง'

df_r['ภาค'] = df_r['PROV_ID'].apply(assign_region)

region_monthly = df_r.groupby(['ภาค','MONTH','MONTH_TH'])['AvgRain'].mean().reset_index()
region_monthly = region_monthly.sort_values('MONTH')

fig = px.line(
    region_monthly, x='MONTH_TH', y='AvgRain',
    color='ภาค', markers=True,
    title='🗺️ ปริมาณฝนเฉลี่ยรายเดือน แยกตามภาค',
    labels={'AvgRain': 'AvgRain เฉลี่ย (mm)', 'MONTH_TH': 'เดือน'},
    template='plotly_white'
)
fig.update_layout(height=450)
fig.show()

region_avg = df_r.groupby('ภาค')['AvgRain'].mean().sort_values(ascending=False)
print(f"""
💡 Interpretation:
   • ภาคที่มีฝนเฉลี่ยสูงสุด: {region_avg.index[0]} ({region_avg.iloc[0]:.1f} mm)
   • ภาคที่มีฝนเฉลี่ยต่ำสุด: {region_avg.index[-1]} ({region_avg.iloc[-1]:.1f} mm)
   • ภาคใต้มีรูปแบบฝนต่างจากภาคอื่น: สูงช่วงปลายปี (ก.ย.–ธ.ค.)
   • ภาคตะวันออกเฉียงเหนือฝนสูงเฉพาะกลางปี (มิ.ย.–ก.ย.) แล้วหยุดทันที
""")


💡 Interpretation:
   • ภาคที่มีฝนเฉลี่ยสูงสุด: ภาคใต้ (203.6 mm)
   • ภาคที่มีฝนเฉลี่ยต่ำสุด: ภาคตะวันตก (91.6 mm)
   • ภาคใต้มีรูปแบบฝนต่างจากภาคอื่น: สูงช่วงปลายปี (ก.ย.–ธ.ค.)
   • ภาคตะวันออกเฉียงเหนือฝนสูงเฉพาะกลางปี (มิ.ย.–ก.ย.) แล้วหยุดทันที



In [ ]:
# ─── 5.9 ตำบลเสี่ยงสูง: COUNT 17 YEAR Distribution ───
high_risk_df = df_f[df_f['RISK']=='เสี่ยงสูง']

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Distribution of COUNT 17 YEAR (เสี่ยงสูง)', 'Count รายจังหวัด Top 15'))

fig.add_trace(
    go.Histogram(x=high_risk_df['COUNT 17 YEAR'], nbinsx=9,
                 marker_color=COLOR_RED, name='Count'),
    row=1, col=1)

top_prov_count = (
    high_risk_df.groupby('PROV_T')['COUNT 17 YEAR'].mean()
    .reset_index().sort_values('COUNT 17 YEAR', ascending=False).head(15)
)
fig.add_trace(
    go.Bar(x=top_prov_count['COUNT 17 YEAR'], y=top_prov_count['PROV_T'],
           orientation='h', marker_color=COLOR_RED, name='avg count'),
    row=1, col=2)

fig.update_layout(
    title='🔴 วิเคราะห์ตำบลเสี่ยงสูง: ความถี่น้ำท่วม 17 ปี',
    height=450, template='plotly_white', showlegend=False
)
fig.update_yaxes(categoryorder='total ascending', row=1, col=2)
fig.show()

print(f"""
💡 Interpretation:
   • ตำบลเสี่ยงสูง = ท่วม 9–17 ครั้งใน 17 ปี (~ทุกปีหรือเกือบทุกปี)
   • จังหวัดอันดับต้นๆ ในกลุ่มนี้ต้องการมาตรการถาวร ไม่ใช่แค่การเตือนภัย
   • จำนวน COUNT สูง = พื้นที่น้ำท่วมซ้ำซาก ควรพิจารณาการวางผังเมืองและระบบป้องกัน
""")


💡 Interpretation:
   • ตำบลเสี่ยงสูง = ท่วม 9–17 ครั้งใน 17 ปี (~ทุกปีหรือเกือบทุกปี)
   • จังหวัดอันดับต้นๆ ในกลุ่มนี้ต้องการมาตรการถาวร ไม่ใช่แค่การเตือนภัย
   • จำนวน COUNT สูง = พื้นที่น้ำท่วมซ้ำซาก ควรพิจารณาการวางผังเมืองและระบบป้องกัน



In [ ]:
# ─── 5.10 Sunburst Chart: ภาค → จังหวัด → ระดับความเสี่ยง ───
df_f['ภาค'] = df_f['PROV_CODE'].apply(lambda x: assign_region(str(x).zfill(2)))

sunburst_df = (
    df_f.groupby(['ภาค','PROV_T','RISK'])['GEOCODE'].nunique()
    .reset_index()
)
sunburst_df.columns = ['ภาค','จังหวัด','RISK','จำนวนตำบล']

fig = px.sunburst(
    sunburst_df.head(200),  # จำกัดเพื่อ performance
    path=['ภาค','จังหวัด','RISK'],
    values='จำนวนตำบล',
    color='RISK',
    color_discrete_map=RISK_COLORS,
    title='🌐 Sunburst: สัดส่วนตำบลตามระดับความเสี่ยง แยกภาค → จังหวัด',
    template='plotly_white'
)
fig.update_layout(height=550)
fig.show()

print("""
💡 Interpretation:
   • คลิกที่ภาคหรือจังหวัดเพื่อ Zoom In ดูรายละเอียด
   • ขนาด segment = จำนวนตำบลในกลุ่มนั้น
   • สีแดง (เสี่ยงสูง) กระจุกตัวในบางภาคมากกว่าภาคอื่น
""")


💡 Interpretation:
   • คลิกที่ภาคหรือจังหวัดเพื่อ Zoom In ดูรายละเอียด
   • ขนาด segment = จำนวนตำบลในกลุ่มนั้น
   • สีแดง (เสี่ยงสูง) กระจุกตัวในบางภาคมากกว่าภาคอื่น



---
# 6️⃣ Dashboard Preparation

### Export ข้อมูลสำหรับ Looker Studio

รวมข้อมูลทั้งหมดเป็นไฟล์เดียว พร้อม Feature ที่คำนวณแล้ว


In [ ]:
# ─── สร้าง Master Table สำหรับ Looker Studio ───

# ─── ไฟล์ที่ 1: ฝนระดับจังหวัด ───
df_export_rain = df_r[[
    'YEAR', 'MONTH', 'MONTH_TH', 'YEAR_MONTH', 'DATE',
    'PROV_ID', 'PROV_T', 'PROV_EN', 'ภาค_REF',
    'MinRain', 'MaxRain', 'AvgRain', 'RAIN_RANGE'
]].rename(columns={'ภาค_REF': 'ภาค'}).copy()

print(f'📦 ไฟล์ฝน: {df_export_rain.shape[0]:,} แถว × {df_export_rain.shape[1]} คอลัมน์')
print(f'   Granularity: จังหวัด ({df_export_rain["PROV_T"].nunique()}) × เดือน × ปี')
print(f'   Null: {df_export_rain.isnull().sum().sum()} ค่า')
display(df_export_rain.head(3))

print()

# ─── ไฟล์ที่ 2: ความเสี่ยงน้ำท่วมระดับตำบล ───
df_export_flood = df_f[[
    'Month', 'MONTH_TH',
    'GEOCODE', 'TAMBON_T', 'TAMBON_E',
    'AMPHOE_CODE', 'AMPHOE_T', 'AMPHOE_E',
    'PROV_CODE', 'PROV_T', 'PROV_EN', 'ภาค_REF',
    'COUNT 17 YEAR', 'CRITERIA', 'RISK', 'RISK_SCORE'
]].rename(columns={'ภาค_REF': 'ภาค', 'COUNT 17 YEAR': 'COUNT_17_YEAR'}).copy()

print(f'📦 ไฟล์น้ำท่วม: {df_export_flood.shape[0]:,} แถว × {df_export_flood.shape[1]} คอลัมน์')
print(f'   Granularity: ตำบล ({df_export_flood["GEOCODE"].nunique():,}) × เดือน')
print(f'   Null: {df_export_flood.isnull().sum().sum()} ค่า')
display(df_export_flood.head(3))

📦 ไฟล์ฝน: 7,546 แถว × 13 คอลัมน์
   Granularity: จังหวัด (77) × เดือน × ปี
   Null: 0 ค่า


,YEAR,MONTH,MONTH_TH,YEAR_MONTH,DATE,PROV_ID,PROV_T,PROV_EN,ภาค,MinRain,MaxRain,AvgRain,RAIN_RANGE
0,2018,1,ม.ค.,2018-01,2018-01-01,10,กรุงเทพมหานคร,Bangkok,กลาง,54.299999,257.230011,142.119137,202.930012
1,2018,1,ม.ค.,2018-01,2018-01-01,11,สมุทรปราการ,Samut Prakan,กลาง,76.250000,256.100006,137.302046,179.850006
2,2018,1,ม.ค.,2018-01,2018-01-01,12,นนทบุรี,Nonthaburi,กลาง,38.360001,161.470001,113.433771,123.110001



📦 ไฟล์น้ำท่วม: 27,023 แถว × 16 คอลัมน์
   Granularity: ตำบล (6,435) × เดือน
   Null: 0 ค่า


,Month,MONTH_TH,GEOCODE,TAMBON_T,TAMBON_E,AMPHOE_CODE,AMPHOE_T,AMPHOE_E,PROV_CODE,PROV_T,PROV_EN,ภาค,COUNT_17_YEAR,CRITERIA,RISK,RISK_SCORE
0,1,ม.ค.,120501,ต.ไทรน้อย,Sai Noi,1205,อ.ไทรน้อย,Sai Noi District,12,นนทบุรี,Nonthaburi,กลาง,1,1-3 ครั้งในรอบ 17 ปี,เสี่ยงต่ำ,1
1,1,ม.ค.,120503,ต.หนองเพรางาย,Nong Phrao Ngai,1205,อ.ไทรน้อย,Sai Noi District,12,นนทบุรี,Nonthaburi,กลาง,1,1-3 ครั้งในรอบ 17 ปี,เสี่ยงต่ำ,1
2,1,ม.ค.,120504,ต.ไทรใหญ่,Sai Yai,1205,อ.ไทรน้อย,Sai Noi District,12,นนทบุรี,Nonthaburi,กลาง,1,1-3 ครั้งในรอบ 17 ปี,เสี่ยงต่ำ,1


In [ ]:
# ─── Export ทั้ง 2 ไฟล์ ───
OUTPUT_RAIN  = 'looker_rain_province.csv'
OUTPUT_FLOOD = 'looker_flood_tambon.csv'

df_export_rain.to_csv(OUTPUT_RAIN,   index=False, encoding='utf-8-sig')
df_export_flood.to_csv(OUTPUT_FLOOD, index=False, encoding='utf-8-sig')

print('💾 Export สำเร็จ!')
print(f'   📄 {OUTPUT_RAIN:<30}: {os.path.getsize(OUTPUT_RAIN)/1024:.1f} KB  |  {df_export_rain.shape[0]:,} แถว')
print(f'   📄 {OUTPUT_FLOOD:<30}: {os.path.getsize(OUTPUT_FLOOD)/1024:.1f} KB  |  {df_export_flood.shape[0]:,} แถว')

print('\n📋 Data Dictionary — looker_rain_province.csv')
rain_dict = {
    'YEAR':       'ปี (2018–2026)',
    'MONTH':      'เดือน (1–12)',
    'MONTH_TH':   'ชื่อเดือนภาษาไทย',
    'YEAR_MONTH': 'YYYY-MM สำหรับ Filter',
    'DATE':       'วันที่แรกของเดือน → ใช้เป็น Date field ใน Looker Studio',
    'PROV_ID':    'รหัสจังหวัด 2 หลัก',
    'PROV_T':     'ชื่อจังหวัดภาษาไทย → Join key กับไฟล์น้ำท่วม',
    'PROV_EN':    'ชื่อจังหวัดภาษาอังกฤษ',
    'ภาค':        'ภูมิภาค (กลาง/เหนือ/อีสาน/ตะวันออก/ใต้)',
    'MinRain':    'ปริมาณฝนต่ำสุดในเดือน (mm)',
    'MaxRain':    'ปริมาณฝนสูงสุดในเดือน (mm)',
    'AvgRain':    'ปริมาณฝนเฉลี่ยในเดือน (mm)',
    'RAIN_RANGE': 'ช่วงฝน MaxRain - MinRain (mm)',
}
for col, desc in rain_dict.items():
    print(f'  {col:<12}: {desc}')

print('\n📋 Data Dictionary — looker_flood_tambon.csv')
flood_dict = {
    'Month':         'เดือน (1–12)',
    'MONTH_TH':      'ชื่อเดือนภาษาไทย',
    'GEOCODE':       'รหัสตำบล 6 หลัก (Unique ID ระดับตำบล)',
    'TAMBON_T':      'ชื่อตำบลภาษาไทย',
    'TAMBON_E':      'ชื่อตำบลภาษาอังกฤษ',
    'AMPHOE_CODE':   'รหัสอำเภอ 4 หลัก',
    'AMPHOE_T':      'ชื่ออำเภอภาษาไทย',
    'AMPHOE_E':      'ชื่ออำเภอภาษาอังกฤษ',
    'PROV_CODE':     'รหัสจังหวัด 2 หลัก',
    'PROV_T':        'ชื่อจังหวัดภาษาไทย → Join key กับไฟล์ฝน',
    'PROV_EN':       'ชื่อจังหวัดภาษาอังกฤษ',
    'ภาค':           'ภูมิภาค',
    'COUNT_17_YEAR': 'จำนวนครั้งที่น้ำท่วมใน 17 ปี (1–17)',
    'CRITERIA':      'เกณฑ์ความถี่ (1-3 / 4-8 / 9-17 ครั้ง)',
    'RISK':          'ระดับความเสี่ยง (เสี่ยงต่ำ/ปานกลาง/สูง)',
    'RISK_SCORE':    'คะแนนความเสี่ยง (1=ต่ำ, 2=ปานกลาง, 3=สูง)',
}
for col, desc in flood_dict.items():
    print(f'  {col:<16}: {desc}')

💾 Export สำเร็จ!
   📄 looker_rain_province.csv      : 986.3 KB  |  7,546 แถว
   📄 looker_flood_tambon.csv       : 6538.9 KB  |  27,023 แถว

📋 Data Dictionary — looker_rain_province.csv
  YEAR        : ปี (2018–2026)
  MONTH       : เดือน (1–12)
  MONTH_TH    : ชื่อเดือนภาษาไทย
  YEAR_MONTH  : YYYY-MM สำหรับ Filter
  DATE        : วันที่แรกของเดือน → ใช้เป็น Date field ใน Looker Studio
  PROV_ID     : รหัสจังหวัด 2 หลัก
  PROV_T      : ชื่อจังหวัดภาษาไทย → Join key กับไฟล์น้ำท่วม
  PROV_EN     : ชื่อจังหวัดภาษาอังกฤษ
  ภาค         : ภูมิภาค (กลาง/เหนือ/อีสาน/ตะวันออก/ใต้)
  MinRain     : ปริมาณฝนต่ำสุดในเดือน (mm)
  MaxRain     : ปริมาณฝนสูงสุดในเดือน (mm)
  AvgRain     : ปริมาณฝนเฉลี่ยในเดือน (mm)
  RAIN_RANGE  : ช่วงฝน MaxRain - MinRain (mm)

📋 Data Dictionary — looker_flood_tambon.csv
  Month           : เดือน (1–12)
  MONTH_TH        : ชื่อเดือนภาษาไทย
  GEOCODE         : รหัสตำบล 6 หลัก (Unique ID ระดับตำบล)
  TAMBON_T        : ชื่อตำบลภาษาไทย
  TAMBON_E        : ชื่อตำบลภาษาอังกฤษ

In [ ]:
# ─── Download ไฟล์ (Google Colab) ───
try:
    from google.colab import files
    for f_out in [OUTPUT_RAIN, OUTPUT_FLOOD]:
        files.download(f_out)
        print(f'⬇️  กำลัง Download: {f_out}')
except ImportError:
    print(f'📁 ไฟล์อยู่ที่: {os.path.abspath(OUTPUT_RAIN)}')
    print(f'📁 ไฟล์อยู่ที่: {os.path.abspath(OUTPUT_FLOOD)}')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎉 Data Storytelling Notebook เสร็จสมบูรณ์!

📌 ขั้นตอนต่อไปสำหรับ Looker Studio:
   1. Upload ทั้ง 2 ไฟล์ CSV ไปยัง Google Sheets
   2. สร้าง Data Source แยกสำหรับแต่ละไฟล์
   3. ตั้งค่า DATE เป็น Date dimension (ไฟล์ฝน)
   4. ใช้ PROV_T เป็น Geo dimension ทั้งสองไฟล์
   5. Blend Data ด้วย PROV_T + Month เพื่อแสดงฝน + ความเสี่ยงในชาร์ตเดียวกัน
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  กำลัง Download: looker_rain_province.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  กำลัง Download: looker_flood_tambon.csv

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎉 Data Storytelling Notebook เสร็จสมบูรณ์!

📌 ขั้นตอนต่อไปสำหรับ Looker Studio:
   1. Upload ทั้ง 2 ไฟล์ CSV ไปยัง Google Sheets
   2. สร้าง Data Source แยกสำหรับแต่ละไฟล์
   3. ตั้งค่า DATE เป็น Date dimension (ไฟล์ฝน)
   4. ใช้ PROV_T เป็น Geo dimension ทั้งสองไฟล์
   5. Blend Data ด้วย PROV_T + Month เพื่อแสดงฝน + ความเสี่ยงในชาร์ตเดียวกัน
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

